In [20]:
# Load the packages for use
library(tidyverse)
library(peacesciencer)
library(reticulate)
library(countrycode)
library(dplyr)
library(states)
data(gwstates)


Attaching package: ‘states’


The following object is masked from ‘package:readr’:

    parse_date




In [2]:
data(ucdp_acd)
View(ucdp_acd)

conflict_id,year,gwno_a,gwno_a_2nd,gwno_b,gwno_b_2nd,incompatibility,intensity_level,type_of_conflict,start_date,start_prec,start_date2,start_prec2,ep_end,ep_end_date
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<date>,<dbl>,<date>,<dbl>,<dbl>,<date>
11342,2012,750,NA,NA,NA,territory,1,intrastate,1997-05-29,1,2012-11-15,1,1,2012-12-21
11342,2014,750,NA,NA,NA,territory,1,intrastate,1997-05-29,1,2014-07-01,1,1,2014-11-27
11343,1967,651,NA,666,NA,territory,2,interstate,1967-06-05,1,1967-06-05,1,1,1967-06-10
11343,1969,651,NA,666,NA,territory,1,interstate,1967-06-05,1,1969-03-06,1,0,NA
11343,1970,651,NA,666,NA,territory,1,interstate,1967-06-05,1,1969-03-06,1,1,1970-08-07
11343,1973,651,NA,666,NA,territory,2,interstate,1967-06-05,1,1973-10-06,1,1,1973-10-24
11344,2011,625,NA,NA,NA,territory,1,intrastate,2011-05-01,1,2011-05-19,1,1,2011-06-15
11345,2011,626,NA,NA,NA,government,1,intrastate,2011-08-20,2,2011-08-20,2,0,NA
11345,2012,626,NA,NA,NA,government,1,intrastate,2011-08-20,2,2011-08-20,2,0,NA


In [ ]:
base <- create_stateyears(system = "gw", subset_years = c(1946:2023)) %>%
  add_ccode_to_gw() %>%
  left_join(
    gwstates %>% select(gwcode, gwc, start, end),
    by = "gwcode",
    relationship = "many-to-many"
  ) %>%
  filter(year >= year(start) & year <= year(end)) %>%
  select(-start, -end)%>%
  mutate(iso3c = countrycode(gwcode, "gwn", "iso3c"))
 
base <- base %>%
  mutate(
    gw_name = case_when(
      gwcode == 403 ~ "Sao Tome and Principe",  # São Tomé and Príncipe
      gwcode == 437 ~ "Ivory Coast",                     # Côte d'Ivoire
      TRUE ~ gw_name
    )
  )

View(base)


Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `iso3c = countrycode(gwcode, "gwn", "iso3c")`.
Caused by warning:
! Some values were not matched unambiguously: 54, 55, 56, 57, 58, 60, 221, 223, 232, 265, 315, 331, 345, 347, 396, 397, 403, 511, 591, 678, 680, 711, 816, 817, 935, 970, 971, 972, 973, 983, 986, 987, 990
To fix unmatched values, please use the `custom_match` argument. If you think the default matching rules should be improved, please file an issue at https://github.com/vincentarelbundock/countrycode/issues”


gwcode,gw_name,microstate,year,ccode,gwc,iso3c
<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
2,United States of America,0,1946,2,USA,USA
2,United States of America,0,1947,2,USA,USA
2,United States of America,0,1948,2,USA,USA
2,United States of America,0,1949,2,USA,USA
2,United States of America,0,1950,2,USA,USA
2,United States of America,0,1951,2,USA,USA
2,United States of America,0,1952,2,USA,USA
2,United States of America,0,1953,2,USA,USA
2,United States of America,0,1954,2,USA,USA


In [37]:
# --- Step 2: living microstates countrycode just misses (safe hard-code) ---
manual_iso3 <- c(
  # microstates countrycode just misses
  "54"  = "DMA", "55" = "GRD", "56" = "LCA", "57" = "VCT", "58" = "ATG", "60" = "KNA",
  "221" = "MCO", "223" = "LIE", "232" = "AND", "331" = "SMR", "591" = "SYC",
  "935" = "VUT", "970" = "KIR", "971" = "NRU", "972" = "TON", "973" = "TUV",
  "983" = "MHL", "986" = "PLW", "987" = "FSM", "990" = "WSM",
  "403" = "STP", "437" = "CIV",
  # historical states' own period ISO3 (not a successor mapping)
  "315" = "CSK",  # Czechoslovakia
  "265" = "DDR",  # East Germany
  "345" = "YUG",  # Yugoslavia (SFRY)
  "680" = "YMD",  # South Yemen (People's Democratic Republic)
  "678" = "YEM",  # North Yemen (Yemen Arab Republic) — kept the continuing code
  "816" = "VDR",  # North Vietnam (Democratic Republic) — distinct retired code
  "817" = "VNM",   # South Vietnam (Republic of) — kept the continuing code
  "347" = "KSV"   # Kosovo
)

base <- base %>%
  mutate(iso3c = if_else(
    is.na(iso3c) & as.character(gwcode) %in% names(manual_iso3),
    manual_iso3[as.character(gwcode)],
    iso3c
  ))

# --- Step 3: historical/dissolved states — no living ISO3, keep iso3c as NA, 
#     but add a SEPARATE successor_iso3c column for merges that need one ---
successor_map <- c(
  "265" = "DEU",  # East Germany -> Germany
  "260" = "DEU",  # West Germany -> Germany
  "315" = "CZE",  # Czechoslovakia -> Czechia (judgment call; could split CZE/SVK by year if needed)
  "345" = "SRB",  # Yugoslavia -> Serbia (Miller's convention; revisit if your DV is territorial)
  "680" = "YEM",  # South Yemen -> Yemen
  "711" = "CHN",  # Tibet -> China (if relevant to your scope)
  "817" = "VNM",  # South Vietnam -> Vietnam
  "347" = "XKX"   # Kosovo -- only if your other sources use this convention
)

base <- base %>%
  mutate(successor_iso3c = if_else(
    as.character(gwcode) %in% names(successor_map),
    successor_map[as.character(gwcode)],
    NA_character_
  ))

# --- Step 5: final check ---
base %>% filter(is.na(iso3c)) %>% distinct(gwcode, gw_name)
# should only show: Abkhazia (396), South Ossetia (397), and any others 
# with genuinely no ISO3/successor convention — leave these as NA  

gwcode,gw_name
<dbl>,<chr>
396,Abkhazia
397,South Ossetia
511,Zanzibar
711,Tibet


In [38]:
write.csv(base, "base.csv", row.names = FALSE)

In [30]:
# --- Step 2: check what's left unmatched ---
base %>%
  distinct(gwcode, gw_name, iso3c) %>%
  filter(is.na(iso3c)) %>%
  arrange(gwcode)

gwcode,gw_name,iso3c
<dbl>,<chr>,<chr>
54,Dominica,NA
55,Grenada,NA
56,Saint Lucia,NA
57,Saint Vincent and the Grenadines,NA
58,Antigua & Barbuda,NA
60,Saint Kitts and Nevis,NA
221,Monaco,NA
223,Liechtenstein,NA
232,Andorra,NA


In [17]:
base %>%
  distinct(gwcode, gwc, iso3c) %>%
  filter(is.na(iso3c) | is.na(gwc)) %>%
  arrange(gwcode)

gwcode,gwc,iso3c
<dbl>,<chr>,<chr>
54,NA,NA
55,NA,NA
56,NA,NA
57,NA,NA
58,NA,NA
60,NA,NA
221,NA,NA
223,NA,NA
232,NA,NA


In [42]:
# all types pooled
ucdp_acd %>%
  group_by(gwno_a, year) %>%
  summarize(n_conflicts_all = n_distinct(conflict_id)) %>%
  rename(gwcode = gwno_a) -> conflict_counts_all

# intrastate + II
ucdp_acd %>%
  filter(type_of_conflict %in% c(3,4)) %>%
  group_by(gwno_a, year) %>%
  summarize(n_conflicts_intra = n_distinct(conflict_id)) %>%
  rename(gwcode = gwno_a) -> conflict_counts_intra

# interstate
ucdp_acd %>%
  filter(type_of_conflict == 2) %>%
  group_by(gwno_a, year) %>%
  summarize(n_conflicts_inter = n_distinct(conflict_id)) %>%
  rename(gwcode = gwno_a) -> conflict_counts_inter

# extrasystemic
ucdp_acd %>%
  filter(type_of_conflict == 1) %>%
  group_by(gwno_a, year) %>%
  summarize(n_conflicts_extra = n_distinct(conflict_id)) %>%
  rename(gwcode = gwno_a) -> conflict_counts_extra

# join everything onto your state-year frame
final <- base %>%
  left_join(conflict_counts_all, by = c("gwcode", "year")) %>%
  left_join(conflict_counts_intra, by = c("gwcode", "year")) %>%
  left_join(conflict_counts_inter, by = c("gwcode", "year")) %>%
  left_join(conflict_counts_extra, by = c("gwcode", "year")) %>%
  mutate(across(starts_with("n_conflicts"), ~replace_na(.x, 0)))

final  

`summarise()` has grouped output by 'gwno_a'. You can override using the
`.groups` argument.
`summarise()` has grouped output by 'gwno_a'. You can override using the
`.groups` argument.
`summarise()` has grouped output by 'gwno_a'. You can override using the
`.groups` argument.
`summarise()` has grouped output by 'gwno_a'. You can override using the
`.groups` argument.


gwcode,gw_name,microstate,year,ccode,gwc,iso3c,successor_iso3c,n_conflicts_all,n_conflicts_intra,n_conflicts_inter,n_conflicts_extra
<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>
2,United States of America,0,1946,2,USA,USA,NA,0,0,0,0
2,United States of America,0,1947,2,USA,USA,NA,0,0,0,0
2,United States of America,0,1948,2,USA,USA,NA,0,0,0,0
2,United States of America,0,1949,2,USA,USA,NA,0,0,0,0
2,United States of America,0,1950,2,USA,USA,NA,1,0,0,0
2,United States of America,0,1951,2,USA,USA,NA,0,0,0,0
2,United States of America,0,1952,2,USA,USA,NA,0,0,0,0
2,United States of America,0,1953,2,USA,USA,NA,0,0,0,0
2,United States of America,0,1954,2,USA,USA,NA,0,0,0,0


In [44]:
final <- final %>%
add_democracy() %>%
add_sim_gdp_pop()

final

Joining with `by = join_by(gwcode, year, euds, aeuds, polity2, v2x_polyarchy)`
Joining with `by = join_by(gwcode, year, mrgdppc, sd_mrgdppc, pwtrgdp,
sd_pwtrgdp, pwtpop, sd_pwtpop)`


gwcode,gw_name,microstate,year,ccode,gwc,iso3c,successor_iso3c,n_conflicts_all,n_conflicts_intra,⋯,euds,aeuds,polity2,v2x_polyarchy,mrgdppc,sd_mrgdppc,pwtrgdp,sd_pwtrgdp,pwtpop,sd_pwtpop
<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<int>,<int>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2,United States of America,0,1946,2,USA,USA,NA,0,0,⋯,1.293985,0.4862558,9,0.603,15916.61,2494.190,2565007,1075414,142.9422,9.874863
2,United States of America,0,1947,2,USA,USA,NA,0,0,⋯,1.308359,0.5006298,9,0.607,15691.61,2443.672,2524909,1097010,145.6134,10.302882
2,United States of America,0,1948,2,USA,USA,NA,0,0,⋯,1.343539,0.5358093,9,0.599,15634.17,2397.095,2621770,1143350,148.4436,10.571322
2,United States of America,0,1949,2,USA,USA,NA,0,0,⋯,1.330836,0.5231064,9,0.580,15774.79,2415.503,2724009,1187962,150.5542,10.712760
2,United States of America,0,1950,2,USA,USA,NA,1,0,⋯,1.354015,0.5462858,9,0.585,16610.07,2605.984,2831568,1207133,153.5495,10.469628
2,United States of America,0,1951,2,USA,USA,NA,0,0,⋯,1.350228,0.5424987,9,0.611,17386.07,2680.565,3084613,1318417,156.0024,10.514347
2,United States of America,0,1952,2,USA,USA,NA,0,0,⋯,1.391332,0.5836026,9,0.611,17880.30,2794.786,3186280,1313776,158.6386,11.001020
2,United States of America,0,1953,2,USA,USA,NA,0,0,⋯,1.391332,0.5836026,9,0.612,18097.87,2813.028,3280505,1382091,162.0679,11.405280
2,United States of America,0,1954,2,USA,USA,NA,0,0,⋯,1.383708,0.5759780,9,0.611,18287.42,2894.786,3368356,1418385,163.9987,11.371540


In [47]:
final %>% filter(is.na(polity2)) %>% distinct(gwcode, gw_name)
# should only show: Abkhazia (396), South Ossetia (397), and any others 
# with genuinely no ISO3/successor convention — leave these as NA  

gwcode,gw_name
<dbl>,<chr>
2,United States of America
20,Canada
31,Bahamas
40,Cuba
41,Haiti
42,Dominican Republic
51,Jamaica
52,Trinidad and Tobago
53,Barbados


In [45]:
write.csv(final, "final.csv", row.names = FALSE, fileEncoding = "UTF-8")


In [ ]:
final %>%
  select(where(is.character)) %>%
  summarise(across(everything(), ~sum(!validEnc(.))))

gw_name,conflict_ids_all,conflict_ids_intra,conflict_ids_inter
<int>,<int>,<int>,<int>
117,0,0,0


In [ ]:
?add_ucdp_acd(type = 'intrastate')

add_ucdp_acd           package:peacesciencer           R Documentation

_A_d_d _U_C_D_P _A_r_m_e_d _C_o_n_f_l_i_c_t _D_a_t_a _t_o _s_t_a_t_e-_y_e_a_r _d_a_t_a _f_r_a_m_e

_D_e_s_c_r_i_p_t_i_o_n:

     ‘add_ucdp_acd()’ allows you to add UCDP Armed Conflict data to a
     state-year data frame

_U_s_a_g_e:

     add_ucdp_acd(data, type, issue, only_wars = FALSE)
     
_A_r_g_u_m_e_n_t_s:

    data: state-year data frame

    type: the types of armed conflicts the user wants to consider,
          specified as a character vector. Options include
          "extrasystemic", "interstate", "intrastate", and "II". "II"
          is convenience shorthand for "internationalized intrastate".
          If you want just one (say: "intrastate"), then the type you
          want in quotes is sufficient. If you want multiple, wrap it
          in a vector with ‘c()’.

   issue: do you want to subset the data to just different armed
 

In [4]:
install.packages("countrycode")

Installing package into ‘/Users/fyodorshafranov/Library/R/arm64/4.5/library’
(as ‘lib’ is unspecified)




The downloaded binary packages are in
	/var/folders/13/671_v3jd3fb5x5kcvpwmqz800000gn/T//RtmpfkmoYh/downloaded_packages


In [5]:
library(countrycode)
data(codelist_panel)
codelist_panel$year

[1] 1789 1790 1791 1792 1793 1794 1795 1796 1797 1798 1799 1800 1801 1802
   [15] 1803 1804 1805 1806 1807 1808 1809 1810 1811 1812 1813 1814 1815 1816
   [29] 1817 1818 1819 1820 1821 1822 1823 1824 1825 1826 1827 1828 1829 1830
   [43] 1831 1832 1833 1834 1835 1836 1837 1838 1839 1840 1841 1842 1843 1844
   [57] 1845 1846 1847 1848 1849 1850 1851 1852 1853 1854 1855 1856 1857 1858
   [71] 1859 1860 1861 1862 1863 1864 1865 1866 1867 1868 1869 1870 1871 1872
   [85] 1873 1874 1875 1876 1877 1878 1879 1880 1881 1882 1883 1884 1885 1886
   [99] 1887 1888 1889 1890 1891 1892 1893 1894 1895 1896 1897 1898 1899 1900
  [113] 1901 1902 1903 1904 1905 1906 1907 1908 1909 1910 1911 1912 1913 1914
  [127] 1915 1916 1917 1918 1919 1920 1921 1922 1923 1924 1925 1926 1927 1928
  [141] 1929 1930 1931 1932 1933 1934 1935 1936 1937 1938 1939 1940 1941 1942
  [155] 1943 1944 1945 1946 1947 1948 1949 1950 1951 1952 1953 1954 1955 1956
  [169] 1957 1958 1959 1960 1961 1962 1963 1964 1965 1966 1967 1968 1969 1970
  [183] 1971 1972 1973 1974 1975 1976 1977 1978 1979 1980 1981 1982 1983 1984
  [197] 1985 1986 1987 1988 1989 1990 1991 1992 1993 1994 1995 1996 1997 1998
  [211] 1999 2000 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 2011 2012
  [225] 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 1912 1913
  [239] 1914 1915 1916 1917 1918 1919 1920 1921 1922 1923 1924 1925 1926 1927
  [253] 1928 1929 1930 1931 1932 1933 1934 1935 1936 1937 1938 1939 1940 1941
  [267] 1942 1943 1944 1945 1946 1947 1948 1949 1950 1951 1952 1953 1954 1955
  [281] 1956 1957 1958 1959 1960 1961 1962 1963 1964 1965 1966 1967 1968 1969
  [295] 1970 1971 1972 1973 1974 1975 1976 1977 1978 1979 1980 1981 1982 1983
  [309] 1984 1985 1986 1987 1988 1989 1990 1991 1992 1993 1994 1995 1996 1997
  [323] 1998 1999 2000 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 2011
  [337] 2012 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 1816
  [351] 1817 1818 1819 1820 1821 1822 1823 1824 1825 1826 1827 1828 1829 1830
  [365] 1900 1901 1902 1903 1904 1905 1906 1907 1908 1909 1910 1911 1912 1913
  [379] 1914 1915 1916 1917 1918 1919 1920 1921 1922 1923 1924 1925 1926 1927
  [393] 1928 1929 1930 1931 1932 1933 1934 1935 1936 1937 1938 1939 1940 1941
  [407] 1942 1943 1944 1945 1946 1947 1948 1949 1950 1951 1952 1953 1954 1955
  [421] 1956 1957 1958 1959 1960 1961 1962 1963 1964 1965 1966 1967 1968 1969
  [435] 1970 1971 1972 1973 1974 1975 1976 1977 1978 1979 1980 1981 1982 1983
  [449] 1984 1985 1986 1987 1988 1989 1990 1991 1992 1993 1994 1995 1996 1997
  [463] 1998 1999 2000 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 2011
  [477] 2012 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 1993
  [491] 1994 1995 1996 1997 1998 1999 2000 2001 2002 2003 2004 2005 2006 2007
  [505] 2008 2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2020 1900
  [519] 1901 1902 1903 1904 1905 1906 1907 1908 1909 1910 1911 1912 1913 1914
  [533] 1915 1916 1917 1918 1919 1920 1921 1922 1923 1924 1925 1926 1927 1928
  [547] 1929 1930 1931 1932 1933 1934 1935 1936 1937 1938 1939 1940 1941 1942
  [561] 1943 1944 1945 1946 1947 1948 1949 1950 1951 1952 1953 1954 1955 1956
  [575] 1957 1958 1959 1960 1961 1962 1963 1964 1965 1966 1967 1968 1969 1970
  [589] 1971 1972 1973 1974 1975 1976 1977 1978 1979 1980 1981 1982 1983 1984
  [603] 1985 1986 1987 1988 1989 1990 1991 1992 1993 1994 1995 1996 1997 1998
  [617] 1999 2000 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 2011 2012
  [631] 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 1981 1982
  [645] 1983 1984 1985 1986 1987 1988 1989 1990 1991 1992 1993 1994 1995 1996
  [659] 1997 1998 1999 2000 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010
  [673] 2011 2012 2013 2014 2015 2016 2017 2018 2019 2020 1789 1790 1791 1792
  [687] 1793 1794 1795 1796 1797 1798 1799 1800 1801 1802 1803 1804 1805 1806
  [701] 1807 1808 1809 1810 1811 1812 1813 1814 1815 1816 1817 1818 1819 1820
  [715] 1821 1822 1823 182

In [6]:
library(dplyr)
library(tidyr)

codelist_panel_filled <- codelist_panel %>%
  arrange(country.name.en, year) %>%
  group_by(country.name.en) %>%
  fill(everything(), .direction = "down") %>%
  ungroup()

In [7]:
nrow(codelist_panel)

[1] 29574

In [8]:
max_year <- max(codelist_panel_filled$year, na.rm = TRUE)
template <- codelist_panel_filled[codelist_panel_filled$year == max_year, ]

missing_years <- (max_year + 1):2026
extended <- do.call(rbind, lapply(missing_years, function(y) {
  rows <- template
  rows$year <- y
  rows
}))

codelist_panel_ext <- rbind(codelist_panel_filled, extended)
nrow(codelist_panel_ext)  # sanity check: should be nrow(codelist_panel) + 5 * nrow(template)

[1] 29930

In [9]:
nrow(codelist_panel)

[1] 29574

In [10]:
# has iso3c, gwc, gwn, cowc, cown, etc. -- one row per country-YEAR
write.csv(codelist_panel_ext, "codelist_panel.csv", row.names = FALSE)

states

In [11]:
install.packages("states")


Installing package into ‘/Users/fyodorshafranov/Library/R/arm64/4.5/library’
(as ‘lib’ is unspecified)




The downloaded binary packages are in
	/var/folders/13/671_v3jd3fb5x5kcvpwmqz800000gn/T//RtmpfkmoYh/downloaded_packages


In [12]:
install.packages("DT")

Installing package into ‘/Users/fyodorshafranov/Library/R/arm64/4.5/library’
(as ‘lib’ is unspecified)




The downloaded binary packages are in
	/var/folders/13/671_v3jd3fb5x5kcvpwmqz800000gn/T//RtmpfkmoYh/downloaded_packages


In [13]:
library(states)
library(DT)


Attaching package: ‘states’


The following object is masked from ‘package:readr’:

    parse_date




In [ ]:
data(gwstates)

data(cowstates)

HTML widgets cannot be represented in plain text (need html)

In [18]:
write.csv(gwstates, "gwstates.csv", row.names = FALSE)
write.csv(cowstates, "cowstates.csv", row.names = FALSE)

In [19]:
max_year <- max(codelist_panel_ext$year, na.rm = TRUE)  # e.g. 2026

gwstates_prepped <- gwstates %>%
  mutate(
    iso3c = countrycode(country_name, "country.name", "iso3c"),
    start_year = as.integer(format(start, "%Y")),
    end_year   = pmin(as.integer(format(end, "%Y")), max_year)  # cap 9999-12-31 spells
  ) %>%
  filter(!is.na(iso3c), !is.na(start_year), !is.na(end_year))

Warning message:
“There were 2 warnings in `mutate()`.
The first warning was:
ℹ In argument: `iso3c = countrycode(country_name, "country.name", "iso3c")`.
Caused by warning:
! Some values were not matched unambiguously: Abkhazia, Austria-Hungary, Baden, Bavaria, Czechoslovakia, German Democratic Republic, Hanover, Hesse-Darmstadt (Ducal), Hesse-Kassel (Electoral), Kosovo, Mecklenburg-Schwerin, Modena, Orange Free State, Parma, Saxony, South Ossetia, Tibet, Transvaal, Tuscany, Two Sicilies, United Provinces of Central America, Vietnam (Annam/Cochin China/Tonkin), Wurttemberg, Yemen (Arab Republic of Yemen), Yemen, People's Republic of, Yugoslavia, Zanzibar
To fix unmatched values, please use the `custom_match` argument. If you think the default matching rules should be improved, please file an issue at https://github.com/vincentarelbundock/countrycode/issues
ℹ Run `dplyr::last_dplyr_warnings()` to see the 1 remaining warning.”
